# 01. Цены и доходности

**Цель:** получить матрицу доходностей **R** размером T × N (дни × активы). Все дальнейшие шаги — ковариация, PCA, риск портфеля — это операции линейной алгебры над этой матрицей.

**Набор:** 25 крупных американских акций из 5 секторов + TLT (долгосрочные гособлигации США) и GLD (золото) для контраста. Период фиксирован: окт 2021 – сен 2026, чтобы результат воспроизводился.

In [1]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.precision', 3)
from src.data import *
from src.risk import *
from src.pca import *
from src.plots import *

In [2]:
prices = clean_prices(load_prices())   # скорректированные цены (дивиденды и сплиты), кэш в data/prices.csv
print(prices.shape, prices.index.min().date(), '→', prices.index.max().date())
print('пропусков:', int(prices.isna().sum().sum()))
pd.Series({s: ' '.join(t) for s, t in UNIVERSE.items()}).to_frame('тикеры')

(1254, 27) 2021-10-01 → 2026-09-30
пропусков: 0


,тикеры
Tech,AAPL MSFT NVDA GOOGL ORCL
Financials,JPM BAC GS MS WFC
Energy,XOM CVX COP SLB EOG
Healthcare,JNJ PFE MRK UNH ABBV
Staples,PG KO PEP WMT COST
Defensive,TLT GLD


**Решения при очистке:**
- `auto_adjust=True` — без поправки на сплиты в день сплита NVDA 4:1 была бы «доходность» −75%.
- Короткие пропуски заполняются **последней известной** ценой (`ffill`). Интерполяция взяла бы цену из будущего — это look-ahead bias.

## Простые или логарифмические доходности?

| | По **времени** (один актив, много дней) | По **активам** (портфель за один день) |
|---|---|---|
| Лог-доходность $l = \ln(P_t/P_{t-1})$ | ✅ $\ln(P_T/P_0) = \sum_t l_t$ | ❌ $l_p \ne \sum_i w_i l_i$ |
| Простая доходность $r = P_t/P_{t-1} - 1$ | ❌ $1+R = \prod_t (1+r_t)$ | ✅ $r_p = \sum_i w_i r_i = w^\top r$ |

Доходность портфеля — **линейная** функция вектора простых доходностей:
$$r_p = \frac{V_t}{V_{t-1}} - 1 = \sum_i \underbrace{\frac{n_i P_{i,t-1}}{V_{t-1}}}_{w_i}\left(\frac{P_{i,t}}{P_{i,t-1}} - 1\right) = w^\top r$$
Именно из этой линейности следуют $E[r_p] = w^\top\mu$ и $\text{Var}(r_p) = w^\top\Sigma w$. Проверим, что будет, если взвешивать лог-доходности:

In [3]:
r, l = simple_returns(prices), log_returns(prices)
w = np.full(r.shape[1], 1 / r.shape[1])           # равные веса

exact = (prices.iloc[-1] / prices.iloc[0]).to_numpy() @ w - 1
via_log = np.exp(l.sum().to_numpy() @ w) - 1
print(f'5 лет buy&hold, точно:               {exact:+.1%}')
print(f'через взвешивание лог-доходностей:   {via_log:+.1%}')

5 лет buy&hold, точно:               +129.1%
через взвешивание лог-доходностей:   +95.9%


Логарифм вогнутый, поэтому по неравенству Йенсена $\sum_i w_i l_i \le l_p$: взвешенные лог-доходности **занижают** доходность портфеля. За один день разница крошечная, но за 5 лет накапливается до десятков процентных пунктов.

## Лог-доходности не «убирают выбросы»
Логарифм сжимает рост и растягивает падения — распределение становится симметричнее, но хвосты остаются:

In [4]:
extremes = pd.concat([r.stack().nlargest(3), r.stack().nsmallest(3)])
pd.DataFrame({'простая r': extremes, 'лог l': np.log1p(extremes)})

,,простая r,лог l
Date,,,
2025-09-10,ORCL,0.359,0.307
2023-05-25,NVDA,0.244,0.218
2025-04-09,NVDA,0.187,0.172
2025-04-17,UNH,-0.224,-0.253
2026-01-27,UNH,-0.196,-0.218
2025-05-13,UNH,-0.178,-0.196


**Решение для проекта:** простые дневные доходности для Σ, PCA и портфелей (портфельная математика точная); лог-доходности — только для накопления во времени.